# Análise dos Dados

Este notebook utiliza as tabelas disponibilizadas na camada Gold para responder
às perguntas definidas no início do projeto.

Perguntas analisadas:

1. Pokémon com maior velocidade apresentam maior taxa de vitória?
2. Quais atributos apresentam as maiores diferenças entre vencedores e perdedores?
3. Pokémon Lendário apresentam taxa de vitória diferente dos demais?
4. Como a taxa de vitória varia entre as diferentes gerações?
5. Quais Pokémon apresentam as maiores taxas de vitória considerando um número mínimo relevante de batalhas?

#1. Conhecendo a distribuição de batalhas

Antes das perguntas, precisamos justificar o que significa "número mínimo relevante de batalhas".

In [0]:
%sql

SELECT
    MIN(total_battles) AS min_battles,
    ROUND(AVG(total_battles), 2) AS avg_battles,
    percentile_approx(total_battles, 0.25) AS p25,
    percentile_approx(total_battles, 0.50) AS median,
    percentile_approx(total_battles, 0.75) AS p75,
    MAX(total_battles) AS max_battles
FROM mvp_data_engineering.gold.agg_pokemon_performance
WHERE total_battles > 0;

min_battles,avg_battles,p25,median,p75,max_battles
92,127.55,120,128,135,164


Para evitar distorções causadas por Pokémon com poucas observações, foram considerados apenas aqueles com pelo menos 127 batalhas, seguindo a média vista na consulta.

#2. Pergunta 1 — Speed × taxa de vitória

Nossa pergunta:

Pokémon com maior atributo de velocidade (Speed) apresentam maior taxa de vitória?

In [0]:
%sql

WITH speed_groups AS (
    SELECT
        p.pokemon_id,
        p.name,
        p.speed,
        a.total_battles,
        a.win_rate,
        CASE
            WHEN p.speed < 40 THEN '01 - < 40'
            WHEN p.speed < 60 THEN '02 - 40-59'
            WHEN p.speed < 80 THEN '03 - 60-79'
            WHEN p.speed < 100 THEN '04 - 80-99'
            ELSE '05 - >= 100'
        END AS speed_group
    FROM mvp_data_engineering.gold.dim_pokemon p
    INNER JOIN mvp_data_engineering.gold.agg_pokemon_performance a
        ON p.pokemon_id = a.pokemon_id
    WHERE a.total_battles > 0
)

SELECT
    speed_group,
    COUNT(*) AS pokemon_count,
    ROUND(AVG(speed), 2) AS avg_speed,
    ROUND(AVG(win_rate) * 100, 2) AS avg_win_rate_pct
FROM speed_groups
GROUP BY speed_group
ORDER BY speed_group;

speed_group,pokemon_count,avg_speed,avg_win_rate_pct
01 - < 40,127,27.6,17.44
02 - 40-59,186,48.37,30.07
03 - 60-79,178,66.99,49.8
04 - 80-99,157,87.96,68.47
05 - >= 100,136,112.94,87.12


In [0]:
%sql

SELECT
    ROUND(CORR(p.speed, a.win_rate), 4) AS speed_win_rate_correlation
FROM mvp_data_engineering.gold.dim_pokemon p
INNER JOIN mvp_data_engineering.gold.agg_pokemon_performance a
    ON p.pokemon_id = a.pokemon_id
WHERE a.total_battles > 0;

speed_win_rate_correlation
0.9381


Os resultados indicam uma associação positiva entre a velocidade dos Pokémon e suas respectivas taxas de vitória, sugerindo que Pokémon com maior atributo de Speed tendem a apresentar melhor desempenho nos combates analisados.

#3. Pergunta 2 — atributos dos vencedores × perdedores

Essa é a análise mais interessante envolvendo diretamente nossa fato.

Quais atributos apresentam as maiores diferenças entre vencedores e perdedores?



In [0]:
%sql

WITH battle_attributes AS (

    SELECT
        winner.hp AS winner_hp,
        winner.attack AS winner_attack,
        winner.defense AS winner_defense,
        winner.sp_attack AS winner_sp_attack,
        winner.sp_defense AS winner_sp_defense,
        winner.speed AS winner_speed,

        loser.hp AS loser_hp,
        loser.attack AS loser_attack,
        loser.defense AS loser_defense,
        loser.sp_attack AS loser_sp_attack,
        loser.sp_defense AS loser_sp_defense,
        loser.speed AS loser_speed

    FROM mvp_data_engineering.gold.fact_battles f

    INNER JOIN mvp_data_engineering.gold.dim_pokemon winner
        ON f.winner_id = winner.pokemon_id

    INNER JOIN mvp_data_engineering.gold.dim_pokemon loser
        ON f.loser_id = loser.pokemon_id
),

averages AS (

    SELECT
        AVG(winner_hp) AS winner_hp,
        AVG(loser_hp) AS loser_hp,

        AVG(winner_attack) AS winner_attack,
        AVG(loser_attack) AS loser_attack,

        AVG(winner_defense) AS winner_defense,
        AVG(loser_defense) AS loser_defense,

        AVG(winner_sp_attack) AS winner_sp_attack,
        AVG(loser_sp_attack) AS loser_sp_attack,

        AVG(winner_sp_defense) AS winner_sp_defense,
        AVG(loser_sp_defense) AS loser_sp_defense,

        AVG(winner_speed) AS winner_speed,
        AVG(loser_speed) AS loser_speed

    FROM battle_attributes
)

SELECT
    'HP' AS attribute,
    ROUND(winner_hp, 2) AS winner_avg,
    ROUND(loser_hp, 2) AS loser_avg,
    ROUND((winner_hp / loser_hp - 1) * 100, 2) AS difference_pct
FROM averages

UNION ALL

SELECT
    'Attack',
    ROUND(winner_attack, 2),
    ROUND(loser_attack, 2),
    ROUND((winner_attack / loser_attack - 1) * 100, 2)
FROM averages

UNION ALL

SELECT
    'Defense',
    ROUND(winner_defense, 2),
    ROUND(loser_defense, 2),
    ROUND((winner_defense / loser_defense - 1) * 100, 2)
FROM averages

UNION ALL

SELECT
    'Sp. Attack',
    ROUND(winner_sp_attack, 2),
    ROUND(loser_sp_attack, 2),
    ROUND((winner_sp_attack / loser_sp_attack - 1) * 100, 2)
FROM averages

UNION ALL

SELECT
    'Sp. Defense',
    ROUND(winner_sp_defense, 2),
    ROUND(loser_sp_defense, 2),
    ROUND((winner_sp_defense / loser_sp_defense - 1) * 100, 2)
FROM averages

UNION ALL

SELECT
    'Speed',
    ROUND(winner_speed, 2),
    ROUND(loser_speed, 2),
    ROUND((winner_speed / loser_speed - 1) * 100, 2)
FROM averages

ORDER BY difference_pct DESC;

attribute,winner_avg,loser_avg,difference_pct
Speed,82.24,54.26,51.56
Sp. Attack,80.82,64.74,24.83
Attack,87.41,70.74,23.57
Sp. Defense,76.36,67.74,12.72
HP,72.53,65.63,10.51
Defense,75.97,72.43,4.89


Os vencedores apresentaram médias superiores em todos os atributos analisados, com destaque para Speed, cuja média foi 51,56% maior em relação aos perdedores, sendo o atributo com maior diferença observada.

#4. Pergunta 3 — Legendary × não Legendary

Pokémon Lendário apresentam taxa de vitória diferente dos demais?

In [0]:
%sql

WITH participations AS (

    SELECT
        first_pokemon_id AS pokemon_id,
        CASE
            WHEN winner_id = first_pokemon_id THEN 1
            ELSE 0
        END AS win
    FROM mvp_data_engineering.gold.fact_battles

    UNION ALL

    SELECT
        second_pokemon_id AS pokemon_id,
        CASE
            WHEN winner_id = second_pokemon_id THEN 1
            ELSE 0
        END AS win
    FROM mvp_data_engineering.gold.fact_battles
)

SELECT
    p.is_legendary,
    COUNT(*) AS battle_participations,
    SUM(pa.win) AS wins,
    ROUND(
        SUM(pa.win) / COUNT(*) * 100,
        2
    ) AS win_rate_pct

FROM participations pa

INNER JOIN mvp_data_engineering.gold.dim_pokemon p
    ON pa.pokemon_id = p.pokemon_id

GROUP BY p.is_legendary

ORDER BY p.is_legendary;

is_legendary,battle_participations,wins,win_rate_pct
false,91812,43623,47.51
true,8188,6377,77.88


Os Pokémon lendários apresentaram taxa de vitória de 77,88%, frente a 47,51% dos não lendários, indicando desempenho superior desse grupo nos combates analisados.

#5. Pergunta 4 — desempenho por geração

Como a taxa de vitória varia entre as diferentes gerações?

In [0]:
%sql

WITH participations AS (

    SELECT
        first_pokemon_id AS pokemon_id,
        CASE
            WHEN winner_id = first_pokemon_id THEN 1
            ELSE 0
        END AS win
    FROM mvp_data_engineering.gold.fact_battles

    UNION ALL

    SELECT
        second_pokemon_id AS pokemon_id,
        CASE
            WHEN winner_id = second_pokemon_id THEN 1
            ELSE 0
        END AS win
    FROM mvp_data_engineering.gold.fact_battles
)

SELECT
    p.generation,
    COUNT(*) AS battle_participations,
    SUM(pa.win) AS wins,

    ROUND(
        SUM(pa.win) / COUNT(*) * 100,
        2
    ) AS win_rate_pct

FROM participations pa

INNER JOIN mvp_data_engineering.gold.dim_pokemon p
    ON pa.pokemon_id = p.pokemon_id

GROUP BY p.generation

ORDER BY p.generation;

generation,battle_participations,wins,win_rate_pct
1,20311,10513,51.76
2,13310,5966,44.82
3,20388,9708,47.62
4,15137,8160,53.91
5,20607,10662,51.74
6,10247,4991,48.71


A **4ª geração apresentou a maior taxa de vitória (53,91%)**, enquanto a **2ª geração registrou a menor (44,82%)**. Os resultados não indicam uma tendência consistente de aumento ou redução da taxa de vitória entre as gerações.


##Pergunta 5 — Pokémon com maior taxa de vitória

Quais Pokémon apresentam as maiores taxas de vitória considerando um número mínimo relevante de batalhas?

In [0]:
%sql

SELECT
    p.name,
    p.type_1,
    p.type_2,
    p.generation,
    p.is_legendary,
    a.total_battles,
    a.total_wins,
    a.total_losses,
    ROUND(a.win_rate * 100, 2) AS win_rate_pct

FROM mvp_data_engineering.gold.agg_pokemon_performance a

INNER JOIN mvp_data_engineering.gold.dim_pokemon p
    ON a.pokemon_id = p.pokemon_id

WHERE a.total_battles >= 127

ORDER BY
    a.win_rate DESC,
    a.total_battles DESC

LIMIT 20;

name,type_1,type_2,generation,is_legendary,total_battles,total_wins,total_losses,win_rate_pct
Mega Aerodactyl,Rock,Flying,1,false,129,127,2,98.45
Aerodactyl,Rock,Flying,1,false,141,136,5,96.45
Mega Lopunny,Normal,Fighting,4,false,129,124,5,96.12
Greninja,Water,Dark,6,false,127,122,5,96.06
Mega Houndoom,Dark,Fire,2,false,135,128,7,94.81
Mega Rayquaza,Dragon,Flying,3,true,134,127,7,94.78
Mega Salamence,Dragon,Flying,3,false,133,125,8,93.98
Mega Absol,Dark,null,3,false,139,130,9,93.53
Mewtwo,Psychic,null,1,true,164,152,12,92.68
Mega Mewtwo X,Psychic,Fighting,1,true,135,125,10,92.59


Considerando o mínimo de 127 batalhas, o Mega Aerodactyl apresentou a maior taxa de vitória (98,45%), seguido por Aerodactyl (96,45%), Mega Lopunny (96,12%) e Greninja (96,06%). O corte mínimo foi utilizado para evitar que Pokémon com poucas participações distorcessem o ranking.

## Análise dos Resultados

As análises realizadas indicam que **Speed foi o atributo que mais diferenciou vencedores e perdedores**, apresentando média 51,56% superior entre os vencedores e uma associação positiva com a taxa de vitória. Pokémon **lendários** também apresentaram desempenho superior, com taxa de vitória de 77,88%, frente a 47,51% dos demais.

Entre as gerações, a 4ª apresentou a maior taxa de vitória (53,91%) e a 2ª a menor (44,82%), sem evidências de uma tendência consistente de aumento ou redução do desempenho entre gerações. Por fim, considerando apenas Pokémon com pelo menos 127 batalhas, **Mega Aerodactyl liderou o ranking de desempenho, com taxa de vitória de 98,45%**.

De forma consolidada, os resultados destacam principalmente velocidade, atributos ofensivos e a condição de lendário como características associadas ao desempenho nos combates analisados.
